# 02 – Model Evaluation
Python equivalent of `ModelsPerform_1DLSTM_01.m`.

Loads pre-computed results from the `.mat` files and reproduces all performance figures.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from utils import load_eval_results, compute_rmse, compute_r, matlab_datenum_to_datetime
from evaluate import plot_performance, plot_best_models, build_model_meta

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1 · Load results

In [ ]:
MAT_FILE = Path('../data/raw/Models_1DLSTM_2022_02_04_EvalResults.mat')

res   = load_eval_results(MAT_FILE)
YPred = res['YPred']   # (n_timesteps, n_reps, n_models)
YTest = res['YTest']   # (n_timesteps,)
tTest = res['tTest']   # MATLAB datenums

print('YPred shape:', YPred.shape)
print('YTest shape:', YTest.shape)
print('Date range: ', matlab_datenum_to_datetime(tTest[[0,-1]]))

## 2 · Compute metrics

In [ ]:
N_BACK_SAMPLES = [1, 3, 4, 5, 6]
n_mdls_half    = len(N_BACK_SAMPLES)
n_models       = YPred.shape[2]

mean_rmse = np.zeros(n_models)
std_rmse  = np.zeros(n_models)
mean_r    = np.zeros(n_models)
std_r     = np.zeros(n_models)

for i in range(n_models):
    mean_rmse[i], std_rmse[i] = compute_rmse(YPred[:, :, i], YTest)
    mean_r[i],    std_r[i]    = compute_r(   YPred[:, :, i], YTest)

lag_labels, _ = build_model_meta(N_BACK_SAMPLES)

print('Mean RMSE per model:', np.round(mean_rmse, 3))
print('Mean r    per model:', np.round(mean_r,    3))

## 3 · Figure 1 – Performance overview

In [ ]:
plot_performance(
    mean_rmse, std_rmse, mean_r, std_r,
    n_mdls_half=n_mdls_half,
    lag_labels=lag_labels[:n_mdls_half],
    save_path=Path('../outputs/Fig1_model_performance.png'),
)

## 4 · Figure 2 – Best models: D→D and O→D

In [ ]:
cmap     = plt.get_cmap('tab10')

plot_best_models(
    y_pred_dd = YPred[:, :, 0],   # D->D, 1 past sample
    y_pred_od = YPred[:, :, 9],   # O->D, last lag window
    y_test    = YTest,
    t_test    = tTest,
    color_dd  = cmap(0),
    color_od  = cmap(4),
    save_path = Path('../outputs/Fig2_best_models.png'),
)